## SEÇÃO 1 — Imports

Importa scikit-learn para TF-IDF, rank-bm25 para BM25Okapi, scipy para matrizes esparsas e PySpark para o pipeline RDD de agregação por source.

In [1]:
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import os, re, sys, json, random, warnings, unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import pyarrow as pa
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from rank_bm25 import BM25Okapi

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

warnings.filterwarnings("ignore")
import sklearn, rank_bm25 as _rbm25_mod
print(f"Python     : {sys.version.split()[0]}")
print(f"PySpark    : {pyspark.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"rank-bm25  : {_rbm25_mod.__version__ if hasattr(_rbm25_mod,"__version__") else "ok"}")
print(f"Pandas     : {pd.__version__}")

Python     : 3.12.12
PySpark    : 3.5.1
scikit-learn: 1.8.0
rank-bm25  : ok
Pandas     : 2.2.1


## SEÇÃO 2 — Configuração, Paths e SparkSession

Importa constantes do `config/settings.py` (NB00), define `GOLD_TFIDF_DIR` como destino dos outputs e inicia SparkSession com config idêntica aos notebooks anteriores.

In [2]:
PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent
    _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from config.settings import (
    RANDOM_SEED, SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)
from src.utils.logger import ingestion_logger, log_spark_start

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SILVER_DIR       = PROJECT_ROOT / "data" / "silver"
GOLD_WC_DIR      = PROJECT_ROOT / "data" / "gold" / "word_count"
GOLD_TFIDF_DIR   = PROJECT_ROOT / "data" / "gold" / "tfidf_bm25"
GOLD_TFIDF_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_tfidf")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
log_spark_start(logger, spark.sparkContext.appName, spark.version)

print(f"PROJECT_ROOT     : {PROJECT_ROOT}")
print(f"GOLD_TFIDF_DIR   : {GOLD_TFIDF_DIR}")
print(f"SparkSession     : {spark.sparkContext.appName}")

26/06/06 18:02:18 WARN Utils: Your hostname, MacBook-Air-100.local resolves to a loopback address: 127.0.0.1; using 192.168.15.13 instead (on interface en0)
26/06/06 18:02:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 18:02:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/06 18:02:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/06 18:02:19 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


2026-06-06 18:02:20 | INFO     | fii.ingestion                | SPARK_START | app=Investor-Intelligence-Platform-FIIs_tfidf | version=3.5.1
PROJECT_ROOT     : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks
GOLD_TFIDF_DIR   : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/tfidf_bm25
SparkSession     : Investor-Intelligence-Platform-FIIs_tfidf


## SEÇÃO 3 — Leitura do Silver e dos Outputs NB03

Carrega o Silver (NB02) e os parquets Gold do NB03 para enriquecer o output combinado com frequência global, flags TOFU/negativo e risk_level já computados.

In [3]:
# -- Silver
_sp = SILVER_DIR / "silver_articles.parquet"
if not _sp.exists():
    raise FileNotFoundError(f"Silver nao encontrado: {_sp}. Execute NB02.")
df_silver = pd.read_parquet(_sp)
df_silver  = df_silver[df_silver["has_content"] == True].copy()
df_silver["text_clean"] = df_silver["text_clean"].fillna("").astype(str)

# -- NB03 Gold outputs
_gw_path = GOLD_WC_DIR / "global_word_count.parquet"
_nc_path = GOLD_WC_DIR / "negative_context.parquet"
df_global_wc  = pd.read_parquet(_gw_path) if _gw_path.exists() else pd.DataFrame()
df_neg_ctx_nb3= pd.read_parquet(_nc_path) if _nc_path.exists() else pd.DataFrame()

# -- Dicionarios de lookup NB03
_freq_map     = dict(zip(df_global_wc["term"], df_global_wc["frequency"])) \
                if not df_global_wc.empty else {}
_risk_map     = dict(zip(df_neg_ctx_nb3["term"], df_neg_ctx_nb3["risk_level"])) \
                if not df_neg_ctx_nb3.empty else {}
_neg_ratio_map= dict(zip(df_neg_ctx_nb3["term"], df_neg_ctx_nb3["negative_ctx_ratio"])) \
                if not df_neg_ctx_nb3.empty else {}

print(f"Silver (has_content=True) : {len(df_silver):,} artigos")
print(f"NB03 vocabulario          : {len(df_global_wc):,} termos")
print(f"NB03 negative context     : {len(df_neg_ctx_nb3):,} termos com ratio")
print(f"\n  source_type:\n{df_silver['source_type'].value_counts().to_string()}")

Silver (has_content=True) : 122 artigos
NB03 vocabulario          : 8,119 termos
NB03 negative context     : 1,354 termos com ratio

  source_type:
source_type
scraping    63
rss         59


## SEÇÃO 4 — Tokenização do Corpus e Vocabulário TOFU/Negativo

Re-define `tokenize()` (idêntica ao NB03 para reprodutibilidade), pré-tokeniza o corpus Silver e define os dicionários de queries TOFU para o BM25.

In [4]:
# -- Reproduzir tokenizador do NB03 (garantia de consistencia)
for _res in ["stopwords"]:
    try:
        nltk.data.find(f"corpora/{_res}")
    except LookupError:
        nltk.download(_res, quiet=True)
from nltk.corpus import stopwords as _nltk_sw
_SW_PT = set(_nltk_sw.words("portuguese"))
_SW_CUSTOM = {
    "ser","ter","fazer","poder","haver","estar","anos","ano","meses","mes",
    "vez","vezes","forma","modo","tipo","caso","parte","valor","valores",
    "ainda","tambem","pode","sendo","entre","sobre","disso","isso","esse","essa",
    "vai","vao","faz","fez","foi","sao","tem","mais","menos","muito","pouco",
    "novo","novos","nova","novas","real","reais","brasil","brasileiro","mercado",
}
_STOPWORDS = _SW_PT | _SW_CUSTOM
_TOKEN_RE   = re.compile(r"[a-zà-ü]{3,}", re.IGNORECASE | re.UNICODE)

def _norm(s):
    return unicodedata.normalize("NFD", s).encode("ascii","ignore").decode("ascii").lower()

def tokenize(text: str) -> list:
    '''Identico ao NB03 — lowercase + regex unicode PT + stopwords.'''
    if not text or not isinstance(text, str): return []
    tokens = _TOKEN_RE.findall(text.lower())
    return [_norm(t) for t in tokens if _norm(t) not in _STOPWORDS and len(t) >= 3]

# -- Termos TOFU (normalizados)
TOFU_NORM = {
    "dividendos","passiva","patrimonio","recorrencia","diversificacao",
    "yield","inflacao","gestao","carteira","previsibilidade","proventos",
    "valorizacao","fluxo","independencia","imobiliarios","mensal",
    "investimento","fundos","renda","caixa",
}
NEGATIVE_NORM = {
    "vacancia","inadimplencia","risco","prejuizo","corte","alavancagem",
    "queda","crise","desvalorizacao","deterioracao","incerteza",
    "liquidez","reducao","juros","credito","perdas","negativo","baixa",
}

# -- Queries BM25: TOFU (unigram + bigram compostos)
TOFU_QUERIES = {
    "dividendos":              ["dividendos"],
    "renda_passiva":           ["renda", "passiva"],
    "patrimonio":              ["patrimonio"],
    "proventos":               ["proventos"],
    "fundos_imobiliarios":     ["fundos", "imobiliarios"],
    "yield":                   ["yield"],
    "diversificacao":          ["diversificacao"],
    "fluxo_caixa":             ["fluxo", "caixa"],
    "independencia_financeira":["independencia", "financeira"],
    "valorizacao_patrimonial": ["valorizacao", "patrimonial"],
    "renda_mensal":            ["renda", "mensal"],
    "carteira":                ["carteira"],
    "previsibilidade":         ["previsibilidade"],
    "inflacao":                ["inflacao"],
    "recorrencia":             ["recorrencia"],
}

# -- Pre-tokenizacao do corpus (um-unico-passo para TF-IDF e BM25)
print("Pre-tokenizando corpus Silver...")
_token_lists   = [tokenize(t) for t in df_silver["text_clean"]]  # para BM25
_token_strs    = [" ".join(tl) for tl in _token_lists]            # para TF-IDF
_source_labels = df_silver["source_label"].tolist()
_source_types  = df_silver["source_type"].tolist()
_article_ids   = df_silver["article_id"].tolist()

print(f"Corpus tokenizado: {len(_token_lists):,} documentos")
print(f"Tokens medios/doc: {np.mean([len(t) for t in _token_lists]):.1f}")
print(f"TOFU queries     : {len(TOFU_QUERIES)}")

Pre-tokenizando corpus Silver...
Corpus tokenizado: 122 documentos
Tokens medios/doc: 269.7
TOFU queries     : 15


## SEÇÃO 5 — TF-IDF: Fit no Corpus Silver

Ajusta `TfidfVectorizer` com `sublinear_tf=True` (escala log), `ngram_range=(1,2)` e `max_features=10000` sobre o corpus pré-tokenizado. Computa o score médio por termo.

In [5]:
# -- TfidfVectorizer (tokens ja normalizados — token_pattern simples)
vectorizer = TfidfVectorizer(
    max_features   = 10_000,
    ngram_range    = (1, 2),
    min_df         = 2,
    max_df         = 0.90,
    sublinear_tf   = True,          # TF logaritmico: 1 + log(tf)
    analyzer       = "word",
    token_pattern  = r"[a-z]{3,}",  # tokens ja normalizados (sem acento)
)

tfidf_matrix  = vectorizer.fit_transform(_token_strs)
feature_names = vectorizer.get_feature_names_out()

print(f"TF-IDF matrix shape  : {tfidf_matrix.shape}")
print(f"  documentos         : {tfidf_matrix.shape[0]:,}")
print(f"  features (termos)  : {tfidf_matrix.shape[1]:,}")
print(f"  sparsidade         : {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4f}")

# -- Score medio por termo
_mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
df_tfidf_terms = (
    pd.DataFrame({"term": feature_names, "mean_tfidf": _mean_tfidf})
    .sort_values("mean_tfidf", ascending=False)
    .reset_index(drop=True)
)
df_tfidf_terms["tfidf_rank"] = df_tfidf_terms.index + 1
df_tfidf_terms["is_tofu"]    = df_tfidf_terms["term"].isin(TOFU_NORM)
df_tfidf_terms["is_negative"]= df_tfidf_terms["term"].isin(NEGATIVE_NORM)

print(f"\n  Top 20 termos por TF-IDF medio:")
print(f"  {'Rank':<6} {'Termo':<28} {'Mean TF-IDF':>12} {'TOFU':>5} {'NEG':>5}")
print(f"  {'-'*60}")
for _, row in df_tfidf_terms.head(20).iterrows():
    t = "Y" if row["is_tofu"] else ""
    n = "Y" if row["is_negative"] else ""
    print(f"  {int(row['tfidf_rank']):<6} {row['term']:<28} {row['mean_tfidf']:>12.6f} {t:>5} {n:>5}")

TF-IDF matrix shape  : (122, 6597)
  documentos         : 122
  features (termos)  : 6,597
  sparsidade         : 0.9684

  Top 20 termos por TF-IDF medio:
  Rank   Termo                         Mean TF-IDF  TOFU   NEG
  ------------------------------------------------------------
  1      fundos                           0.025095     Y      
  2      fiis                             0.024052            
  3      nao                              0.022266            
  4      imobiliarios                     0.019863     Y      
  5      fundos imobiliarios              0.017833            
  6      acoes                            0.017104            
  7      renda                            0.016578     Y      
  8      dividendos                       0.015967     Y      
  9      carteira                         0.015860     Y      
  10     fundo                            0.015776            
  11     segundo                          0.015708            
  12     todos           

## SEÇÃO 6 — TF-IDF por Source via RDD

Extrai scores TF-IDF dos termos TOFU por documento e usa pipeline RDD (`parallelize -> map -> reduceByKey -> mapValues -> sortBy -> collect`) para agregar média e máximo por source.

In [6]:
# -- Indices dos termos TOFU no vocabulario TF-IDF
_tofu_feat_idx = {
    i: fn for i, fn in enumerate(feature_names)
    if fn in TOFU_NORM
}
_tofu_idx_list  = list(_tofu_feat_idx.keys())
_tofu_feat_list = list(_tofu_feat_idx.values())
print(f"Termos TOFU no vocabulario TF-IDF: {len(_tofu_idx_list)}/{len(TOFU_NORM)}")

# -- Submatriz esparsa TOFU (memory-safe)
tfidf_tofu = tfidf_matrix[:, _tofu_idx_list]   # shape: (n_docs, n_tofu_terms)

# -- Construir lista de tuplas (source, term, score) para o RDD
# Usar dok_matrix para iterar eficientemente sobre elementos nao-zero
_dok = sp.dok_matrix(tfidf_tofu)
_rdd_input = [
    (_source_labels[int(r)], _tofu_feat_list[int(c)], float(v))
    for (r, c), v in _dok.items()
]
print(f"Pares (source, tofu_term, score) nao-zero: {len(_rdd_input):,}")

# ── Pipeline RDD: media TF-IDF por (source, tofu_term) ───────────────────────
rdd_tfidf = spark.sparkContext.parallelize(_rdd_input)

source_term_tfidf = (
    rdd_tfidf
    .map(lambda x: ((x[0], x[1]), (x[2], 1)))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    .mapValues(lambda v: {"avg": round(v[0]/v[1], 6), "doc_count": v[1], "sum": round(v[0], 6)})
    .map(lambda x: (x[0][0], x[0][1], x[1]["avg"], x[1]["doc_count"], x[1]["sum"]))
    .sortBy(lambda x: (x[0], -x[2]))
    .collect()
)

_stfidf_rows = [
    {"source_label": s, "tofu_term": t, "avg_tfidf": avg,
     "doc_count": dc, "tfidf_sum": ts}
    for s, t, avg, dc, ts in source_term_tfidf
]
df_source_tfidf = pd.DataFrame(_stfidf_rows)

print(f"\n  TF-IDF por source-TOFU: {len(df_source_tfidf):,} pares")
print(f"  RDD: parallelize -> map -> reduceByKey -> mapValues -> sortBy -> collect")

# -- Ranking de sources por TF-IDF medio TOFU agregado
df_source_tfidf_rank = (
    df_source_tfidf
    .groupby("source_label")
    .agg(avg_tfidf_tofu=("avg_tfidf","mean"),
         tofu_terms_covered=("tofu_term","count"))
    .sort_values("avg_tfidf_tofu", ascending=False)
    .reset_index()
)
df_source_tfidf_rank["tfidf_rank"] = df_source_tfidf_rank.index + 1

print(f"\n  Source ranking por TF-IDF TOFU:")
print(f"  {'Rank':<5} {'Source':<32} {'Avg TF-IDF TOFU':>16} {'Terms Covered':>14}")
print(f"  {'-'*70}")
for _, row in df_source_tfidf_rank.iterrows():
    print(f"  {int(row['tfidf_rank']):<5} {row['source_label']:<32}"
          f" {row['avg_tfidf_tofu']:>16.6f} {int(row['tofu_terms_covered']):>14}")

Termos TOFU no vocabulario TF-IDF: 18/20
Pares (source, tofu_term, score) nao-zero: 285



  TF-IDF por source-TOFU: 102 pares
  RDD: parallelize -> map -> reduceByKey -> mapValues -> sortBy -> collect

  Source ranking por TF-IDF TOFU:
  Rank  Source                            Avg TF-IDF TOFU  Terms Covered
  ----------------------------------------------------------------------
  1     Clube FII                                0.153393              4
  2     Bora Investir (B3)                       0.088669              3
  3     Status Invest                            0.074852              6
  4     FIIs.com.br                              0.074350              5
  5     E-Investidor Estadao                     0.063850              3
  6     Money Times                              0.062378             13
  7     Eu Quero Investir                        0.062168              7
  8     CNN Brasil Business                      0.059444              7
  9     Funds Explorer                           0.057651              6
  10    XP Conteudos                             0

## SEÇÃO 7 — BM25: Construção do Modelo

Constrói `BM25Okapi` sobre o corpus Silver pré-tokenizado. BM25 usa IDF probabilístico + normalização por comprimento de documento (k1=1.5, b=0.75).

In [7]:
print("Construindo BM25Okapi...")
bm25 = BM25Okapi(_token_lists, k1=1.5, b=0.75)

print(f"BM25 corpus    : {len(_token_lists):,} documentos")
print(f"BM25 avgdl     : {bm25.avgdl:.1f} tokens/doc")
print(f"BM25 k1        : 1.5 (term saturation)")
print(f"BM25 b         : 0.75 (length normalization)")
print(f"BM25 corpus_size: {bm25.corpus_size:,}")

Construindo BM25Okapi...
BM25 corpus    : 122 documentos
BM25 avgdl     : 269.7 tokens/doc
BM25 k1        : 1.5 (term saturation)
BM25 b         : 0.75 (length normalization)
BM25 corpus_size: 122


## SEÇÃO 8 — BM25: Queries com Termos TOFU

Executa cada query TOFU no BM25, captura os top-K scores por artigo e agrega por `source_label` para produzir o ranking de sources por relevância BM25.

In [8]:
_TOP_K = 50    # top-K documentos por query

_bm25_rows = []
for query_label, query_tokens in TOFU_QUERIES.items():
    scores = bm25.get_scores(query_tokens)            # array shape (n_docs,)
    _top_idx = np.argsort(scores)[::-1][:_TOP_K]     # indices dos top-K
    for rank_i, doc_idx in enumerate(_top_idx, 1):
        if scores[doc_idx] <= 0:
            break
        _bm25_rows.append({
            "query_label":   query_label,
            "article_id":    _article_ids[doc_idx],
            "source_label":  _source_labels[doc_idx],
            "source_type":   _source_types[doc_idx],
            "bm25_score":    round(float(scores[doc_idx]), 6),
            "rank_in_query": rank_i,
        })

df_bm25_results = pd.DataFrame(_bm25_rows)
print(f"BM25 query results: {len(df_bm25_results):,} pares (query, article)")

# -- Agregacao por (query, source)
df_bm25_source = (
    df_bm25_results
    .groupby(["query_label", "source_label", "source_type"])
    .agg(
        avg_bm25 =("bm25_score", "mean"),
        max_bm25 =("bm25_score", "max"),
        doc_count=("article_id", "count"),
    )
    .reset_index()
)

# -- Rank por query
df_bm25_source["rank_in_query"] = (
    df_bm25_source
    .groupby("query_label")["avg_bm25"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

print(f"\n  BM25 source aggregation: {len(df_bm25_source):,} linhas")
print("\n  Top 5 sources por query (avg_bm25):")
for qname in list(TOFU_QUERIES.keys())[:4]:
    _q = df_bm25_source[df_bm25_source["query_label"]==qname].nlargest(3,"avg_bm25")
    print(f"  [{qname}]")
    for _, r in _q.iterrows():
        print(f"    {int(r['rank_in_query'])}."
              f" {r['source_label']:<28} bm25={r['avg_bm25']:.4f} docs={int(r['doc_count'])}")

BM25 query results: 257 pares (query, article)

  BM25 source aggregation: 88 linhas

  Top 5 sources por query (avg_bm25):
  [dividendos]
    1. Investidor10                 bm25=3.1007 docs=9
    2. FIIs.com.br                  bm25=2.8059 docs=3
    3. Money Times                  bm25=2.3221 docs=3
  [renda_passiva]
    1. Suno Research                bm25=3.0837 docs=7
    2. Status Invest                bm25=1.9751 docs=1
    3. Bora Investir (B3)           bm25=1.9294 docs=2
  [patrimonio]
    1. Funds Explorer               bm25=2.4058 docs=6
    2. XP Conteudos                 bm25=2.2703 docs=1
    3. Seu Dinheiro                 bm25=1.9313 docs=1
  [proventos]
    1. Investidor10                 bm25=1.9937 docs=1


## SEÇÃO 9 — Ranking Final de Sources: TF-IDF + BM25 Combinados

Normaliza TF-IDF e BM25 para escala [0,1] com MinMaxScaler e combina com pesos `0.4 × TF-IDF + 0.6 × BM25`. Produz o ranking final dos 21 sources para campanha TOFU.

In [9]:
# -- TF-IDF medio por source (ja calculado via RDD)
_tfidf_src = df_source_tfidf_rank.set_index("source_label")["avg_tfidf_tofu"].to_dict()

# -- BM25 medio global por source (media de todas as queries TOFU)
_bm25_src = (
    df_bm25_source
    .groupby("source_label")["avg_bm25"]
    .mean()
    .to_dict()
)

# -- Uniao de todos os sources do Silver
_all_sources = df_silver[["source_label","source_type"]].drop_duplicates()

_combined_rows = []
for _, row in _all_sources.iterrows():
    src = row["source_label"]
    _combined_rows.append({
        "source_label":    src,
        "source_type":     row["source_type"],
        "avg_tfidf_tofu":  _tfidf_src.get(src, 0.0),
        "avg_bm25_tofu":   _bm25_src.get(src, 0.0),
    })

df_src_ranking = pd.DataFrame(_combined_rows)

# -- MinMax normalization (0-1)
_scaler = MinMaxScaler()
_scores  = _scaler.fit_transform(
    df_src_ranking[["avg_tfidf_tofu","avg_bm25_tofu"]]
)
df_src_ranking["tfidf_norm"] = _scores[:, 0]
df_src_ranking["bm25_norm"]  = _scores[:, 1]

# -- Combined score: 0.4 * TF-IDF + 0.6 * BM25
df_src_ranking["combined_score"] = (
    0.4 * df_src_ranking["tfidf_norm"] +
    0.6 * df_src_ranking["bm25_norm"]
).round(6)

df_src_ranking = (
    df_src_ranking
    .sort_values("combined_score", ascending=False)
    .reset_index(drop=True)
)
df_src_ranking["final_rank"] = df_src_ranking.index + 1

print("  RANKING FINAL DOS 21 SOURCES (TF-IDF 40% + BM25 60%)")
print(f"  {'Rank':<5} {'Source':<32} {'TF-IDF norm':>11} {'BM25 norm':>10} {'Combined':>9}")
print(f"  {'-'*72}")
for _, row in df_src_ranking.iterrows():
    print(
        f"  {int(row['final_rank']):<5} {row['source_label']:<32}"
        f" {row['tfidf_norm']:>11.4f} {row['bm25_norm']:>10.4f}"
        f" {row['combined_score']:>9.4f}"
    )

  RANKING FINAL DOS 21 SOURCES (TF-IDF 40% + BM25 60%)
  Rank  Source                           TF-IDF norm  BM25 norm  Combined
  ------------------------------------------------------------------------
  1     Clube FII                             1.0000     1.0000    1.0000
  2     Money Times                           0.4067     0.9621    0.7399
  3     FIIs.com.br                           0.4847     0.8567    0.7079
  4     XP Conteudos                          0.3620     0.9285    0.7019
  5     CNN Brasil Business                   0.3875     0.8899    0.6890
  6     Funds Explorer                        0.3758     0.8743    0.6749
  7     Status Invest                         0.4880     0.7699    0.6572
  8     Suno Research                         0.3129     0.8834    0.6552
  9     InfoMoney                             0.3149     0.8659    0.6455
  10    Eu Quero Investir                     0.4053     0.7924    0.6375
  11    Bora Investir (B3)                    0.5780    

## SEÇÃO 10 — Relevância Combinada por Termo

Constrói o DataFrame mestre por termo unindo: frequência (NB03) + TF-IDF médio + BM25 médio (via top-K scores) + flags TOFU/negativo/risk_level do NB03.

In [10]:
# -- BM25 score medio por unigram TOFU
_bm25_term_avg = {}
for qname, qtokens in TOFU_QUERIES.items():
    _sc = bm25.get_scores(qtokens)
    _bm25_term_avg[qname] = float(np.mean(np.sort(_sc)[::-1][:_TOP_K]))

# -- Merge: tfidf_terms + freq_map + risk_map
df_combined = df_tfidf_terms.copy()
df_combined["freq_global"]    = df_combined["term"].map(_freq_map).fillna(0).astype(int)
df_combined["risk_level"]     = df_combined["term"].map(_risk_map).fillna("NA")
df_combined["neg_ctx_ratio"]  = df_combined["term"].map(_neg_ratio_map).fillna(0.0)

# -- BM25 score medio para termos TOFU (query name pode ser unigram ou bigram)
def _tofu_bm25(term):
    if term in _bm25_term_avg:
        return _bm25_term_avg[term]
    for qname, qtok in TOFU_QUERIES.items():
        if term in qtok:
            return _bm25_term_avg.get(qname, 0.0)
    return 0.0

df_combined["avg_bm25"]   = df_combined["term"].apply(_tofu_bm25)

# -- Normalizar TF-IDF e BM25 separadamente para combined_score do termo
_sc2 = MinMaxScaler().fit_transform(
    df_combined[["mean_tfidf","avg_bm25"]].fillna(0)
)
df_combined["tfidf_norm"] = _sc2[:, 0]
df_combined["bm25_norm"]  = _sc2[:, 1]
df_combined["combined_score"] = (
    0.4 * df_combined["tfidf_norm"] + 0.6 * df_combined["bm25_norm"]
).round(6)

df_combined = df_combined.sort_values("combined_score", ascending=False).reset_index(drop=True)
df_combined["combined_rank"] = df_combined.index + 1

print(f"Term relevance combined: {len(df_combined):,} termos")
print("\n  Top 20 termos por combined_score:")
print(f"  {'Rank':<5} {'Termo':<26} {'TF-IDF':>8} {'BM25':>8} {'Combined':>9} {'TOFU':>5} {'Risk':>6}")
print(f"  {'-'*70}")
for _, row in df_combined.head(20).iterrows():
    t = "Y" if row["is_tofu"] else ""
    print(
        f"  {int(row['combined_rank']):<5} {row['term']:<26}"
        f" {row['tfidf_norm']:>8.4f} {row['bm25_norm']:>8.4f}"
        f" {row['combined_score']:>9.4f} {t:>5} {row['risk_level']:>6}"
    )

Term relevance combined: 6,597 termos

  Top 20 termos por combined_score:
  Rank  Termo                        TF-IDF     BM25  Combined  TOFU   Risk
  ----------------------------------------------------------------------
  1     fundos                       1.0000   1.0000    1.0000     Y   ALTO
  2     imobiliarios                 0.7878   1.0000    0.9151     Y   ALTO
  3     renda                        0.6546   0.6264    0.6377     Y   ALTO
  4     dividendos                   0.6298   0.5901    0.6060     Y   ALTO
  5     carteira                     0.6254   0.5281    0.5670     Y   ALTO
  6     mensal                       0.1591   0.7724    0.5270     Y   ALTO
  7     patrimonial                  0.2692   0.5011    0.4083         ALTO
  8     passiva                      0.0543   0.6264    0.3975     Y   ALTO
  9     fiis                         0.9577   0.0000    0.3831         ALTO
  10    nao                          0.8853   0.0000    0.3541         ALTO
  11    financei

## SEÇÃO 11 — Escrita da Camada Gold

Persiste cinco parquets em `data/gold/tfidf_bm25/` e o relatório JSON de rastreabilidade do NB04, prontos para leitura pelo NB05 (sentiment) e NB06 (marketing).

In [11]:
_NOW = datetime.now(timezone.utc).isoformat()

# -- 1. TF-IDF por termo
_p1 = GOLD_TFIDF_DIR / "tfidf_term_scores.parquet"
df_tfidf_terms.to_parquet(_p1, index=False, engine="pyarrow")
print(f"[1] {_p1.name}: {len(df_tfidf_terms):,} linhas")

# -- 2. TF-IDF por source (output do RDD)
_p2 = GOLD_TFIDF_DIR / "tfidf_source_scores.parquet"
df_source_tfidf.to_parquet(_p2, index=False, engine="pyarrow")
print(f"[2] {_p2.name}: {len(df_source_tfidf):,} linhas")

# -- 3. BM25 query results
_p3 = GOLD_TFIDF_DIR / "bm25_query_results.parquet"
df_bm25_source.to_parquet(_p3, index=False, engine="pyarrow")
print(f"[3] {_p3.name}: {len(df_bm25_source):,} linhas")

# -- 4. Source ranking combinado (output principal para NB06)
_p4 = GOLD_TFIDF_DIR / "source_relevance_ranking.parquet"
df_src_ranking.to_parquet(_p4, index=False, engine="pyarrow")
print(f"[4] {_p4.name}: {len(df_src_ranking):,} linhas")

# -- 5. Term relevance combined (output principal para NB05/NB06)
_p5 = GOLD_TFIDF_DIR / "term_relevance_combined.parquet"
df_combined.to_parquet(_p5, index=False, engine="pyarrow")
print(f"[5] {_p5.name}: {len(df_combined):,} linhas")

# -- Relatorio JSON
_report = {
    "notebook":         "NB04 tfidf_bm25",
    "timestamp":        _NOW,
    "input_silver":     str(SILVER_DIR / "silver_articles.parquet"),
    "input_nb03_gold":  str(GOLD_WC_DIR),
    "output_dir":       str(GOLD_TFIDF_DIR),
    "tfidf_params": {
        "max_features": 10000, "ngram_range": [1,2],
        "min_df": 2, "max_df": 0.90, "sublinear_tf": True,
    },
    "bm25_params": {"k1": 1.5, "b": 0.75, "top_k": _TOP_K},
    "combination_weights": {"tfidf": 0.4, "bm25": 0.6},
    "corpus_stats": {
        "n_documents":   len(df_silver),
        "n_features":    len(feature_names),
        "n_tofu_queries": len(TOFU_QUERIES),
    },
    "rdd_pipeline": "parallelize->map->reduceByKey->mapValues->sortBy->collect",
    "outputs": [
        {"file": _p1.name, "rows": len(df_tfidf_terms)},
        {"file": _p2.name, "rows": len(df_source_tfidf)},
        {"file": _p3.name, "rows": len(df_bm25_source)},
        {"file": _p4.name, "rows": len(df_src_ranking)},
        {"file": _p5.name, "rows": len(df_combined)},
    ],
    "top3_sources": df_src_ranking.head(3)["source_label"].tolist(),
    "top5_tofu_terms_combined": df_combined[df_combined["is_tofu"]].head(5)["term"].tolist(),
}
_rp = GOLD_TFIDF_DIR / "nb04_report.json"
_rp.write_text(json.dumps(_report, indent=2, ensure_ascii=False))
print(f"\nRelatorio: {_rp}")

[1] tfidf_term_scores.parquet: 6,597 linhas
[2] tfidf_source_scores.parquet: 102 linhas
[3] bm25_query_results.parquet: 88 linhas
[4] source_relevance_ranking.parquet: 17 linhas
[5] term_relevance_combined.parquet: 6,597 linhas

Relatorio: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/tfidf_bm25/nb04_report.json


## SEÇÃO 12 — Checks e Validação dos Outputs Gold

Recarrega os cinco parquets do disco, valida colunas obrigatórias e exibe `df.head(5)` de cada output. Confirma que o ranking de sources está completo (21 entries).

In [12]:
print("=" * 70)
print("  CHECKS -- GOLD / tfidf_bm25")
print("=" * 70)

_checks = {
    "tfidf_term_scores.parquet":    ["term","mean_tfidf","tfidf_rank","is_tofu"],
    "tfidf_source_scores.parquet":  ["source_label","tofu_term","avg_tfidf","doc_count"],
    "bm25_query_results.parquet":   ["query_label","source_label","avg_bm25","rank_in_query"],
    "source_relevance_ranking.parquet":["source_label","tfidf_norm","bm25_norm","combined_score","final_rank"],
    "term_relevance_combined.parquet": ["term","mean_tfidf","avg_bm25","combined_score","combined_rank"],
}

for fname, req_cols in _checks.items():
    _df = pd.read_parquet(GOLD_TFIDF_DIR / fname)
    _miss = [c for c in req_cols if c not in _df.columns]
    _ok   = "OK" if not _miss else f"AVISO: {_miss}"
    print(f"\n  [{_ok}] {fname} — {len(_df):,} linhas")
    print(_df[req_cols].head(5).to_string(index=False))

# -- Verificacoes criticas
_src_rank = pd.read_parquet(GOLD_TFIDF_DIR / "source_relevance_ranking.parquet")
print(f"\n  Sources no ranking: {len(_src_rank)} (esperado: ate 21)")
print(f"  #1 source: {_src_rank.iloc[0]['source_label']} (score={_src_rank.iloc[0]['combined_score']:.4f})")
print(f"  #2 source: {_src_rank.iloc[1]['source_label']} (score={_src_rank.iloc[1]['combined_score']:.4f})")
_comb = pd.read_parquet(GOLD_TFIDF_DIR / "term_relevance_combined.parquet")
print(f"\n  Top TOFU term: {_comb[_comb['is_tofu']==True].iloc[0]['term']}")
print("\n  Gold/tfidf_bm25 verificado -- pronto para NB05")

  CHECKS -- GOLD / tfidf_bm25

  [OK] tfidf_term_scores.parquet — 6,597 linhas
               term  mean_tfidf  tfidf_rank  is_tofu
             fundos    0.025095           1     True
               fiis    0.024052           2    False
                nao    0.022266           3    False
       imobiliarios    0.019863           4     True
fundos imobiliarios    0.017833           5    False

  [OK] tfidf_source_scores.parquet — 102 linhas
       source_label       tofu_term  avg_tfidf  doc_count
 Bora Investir (B3)           renda   0.118345          2
 Bora Investir (B3)    investimento   0.084011          2
 Bora Investir (B3)          fundos   0.063651          2
CNN Brasil Business           fluxo   0.089384          1
CNN Brasil Business previsibilidade   0.069298          2

  [OK] bm25_query_results.parquet — 88 linhas
query_label         source_label  avg_bm25  rank_in_query
   carteira            Clube FII  2.129047              1
   carteira E-Investidor Estadao  1.011019 

## SEÇÃO 13 — CRISP-DM Traceability e Encerramento

Exibe o quadro de rastreabilidade CRISP-DM do NB04 e encerra o SparkSession. `data/gold/tfidf_bm25/` está pronto para NB05 (sentiment contextual).

In [13]:
_box = [
    "=" * 78,
    "  NB04 -- TF-IDF + BM25  |  CRISP-DM: Modeling",
    "=" * 78,
    f"  Silver input  : {str(SILVER_DIR / 'silver_articles.parquet')}",
    f"  NB03 Gold in  : {str(GOLD_WC_DIR)}",
    f"  Output dir    : {str(GOLD_TFIDF_DIR)}",
    "-" * 78,
    "  Modelos aplicados:",
    "    TF-IDF : TfidfVectorizer (sublinear_tf, ngram(1,2), max_features=10k)",
    "    BM25   : BM25Okapi (k1=1.5, b=0.75) — queries: 15 termos TOFU",
    "    Combo  : 0.4 * TF-IDF_norm + 0.6 * BM25_norm (MinMaxScaler)",
    "-" * 78,
    "  RDD Pipeline (TF-IDF source aggregation):",
    "    parallelize -> map -> reduceByKey -> mapValues -> sortBy -> collect",
    "-" * 78,
    "  Por que alem da frequencia (NB03)?",
    "    TF-IDF penaliza termos ubiquos (aparecem em todos os docs)",
    "    BM25 normaliza por comprimento — fontes longas nao dominam",
    "    Combined score: ranking justo entre os 21 sources",
    "-" * 78,
    "  Seed: RANDOM_SEED=42 | adaptive.enabled=false | pyarrow engine",
    "  LGPD: corpus publico editorial",
    "-" * 78,
    "  Proximo: NB05 -- Sentiment Contextual  (CAPSTONE / ETAPA 14)",
    "=" * 78,
]
print("\n".join(_box))

_sr = pd.read_parquet(GOLD_TFIDF_DIR / "source_relevance_ranking.parquet")
print(f"\n  Sources rankeados : {len(_sr)}")
print(f"  Features TF-IDF   : {len(feature_names):,}")
print(f"  BM25 avgdl        : {bm25.avgdl:.1f}")
print(f"  TOFU queries      : {len(TOFU_QUERIES)}")
print("\n  NB04 completo -- Gold/tfidf_bm25 pronto para NB05 (CAPSTONE)")

spark.stop()
import sys, os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print("  SparkSession encerrada")

  NB04 -- TF-IDF + BM25  |  CRISP-DM: Modeling
  Silver input  : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/silver/silver_articles.parquet
  NB03 Gold in  : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/word_count
  Output dir    : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/__notebooks/data/gold/tfidf_bm25
------------------------------------------------------------------------------
  Modelos aplicados:
    TF-IDF : TfidfVectorizer (sublinear_tf, ngram(1,2), max_features=10k)
    BM25   : BM25Okapi (k1=1.5, b=0.75) — queries: 15 termos TOFU
    Combo  : 0.4 * TF-IDF_norm + 0.6 * BM25_norm (MinMaxScaler)
------------------------------------------------------------------------------
  RDD Pipeline (TF-IDF source aggregation):
    parallelize -> map -> reduceByKey -> mapValues -> sortBy -> collect
-------------------------------------------------------------------